# PagedAttention in Python (PyTorch)
Welcome to the code-level deep dive into PagedAttention! In this notebook, we will implement a simplified version of the PagedAttention algorithm from scratch using PyTorch. 

We will build:
1. **The Physical KV Cache Memory Pool**: A pre-allocated tensor of blocks.
2. **The Block Table**: A mapping from logical sequence blocks to physical memory blocks.
3. **The PagedAttention Kernel (Simplified)**: A function that computes attention by dynamically fetching non-contiguous blocks.

In [1]:
import torch
import torch.nn.functional as F

# Hyperparameters for our toy example
BLOCK_SIZE = 4       # Number of tokens per block
NUM_BLOCKS = 16      # Total physical blocks in our GPU DRAM pool
HEAD_DIM = 8         # Dimensionality of the attention head

print(f"Total KV Cache Capacity: {NUM_BLOCKS * BLOCK_SIZE} tokens")

Total KV Cache Capacity: 64 tokens


## 1. The Physical KV Cache Pool
In vLLM, memory is pre-allocated as a massive pool of physical blocks. Let's create our physical cache for Keys and Values.

In [2]:
class KVCacheAllocator:
    def __init__(self, num_blocks, block_size, head_dim):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.head_dim = head_dim
        
        # Physical memory pool for Keys and Values
        # Shape: [num_blocks, block_size, head_dim]
        self.k_cache = torch.zeros((num_blocks, block_size, head_dim))
        self.v_cache = torch.zeros((num_blocks, block_size, head_dim))
        
        # Keep track of which physical blocks are free
        self.free_blocks = list(range(num_blocks))[::-1] # Stack of free blocks (15 down to 0)
        
    def allocate_block(self):
        if not self.free_blocks:
            raise MemoryError("Out of GPU Memory! (Needs Swapping)")
        return self.free_blocks.pop()
    
    def free_block(self, block_idx):
        self.free_blocks.append(block_idx)
        
    def write_token(self, physical_block_idx, token_offset, k_tensor, v_tensor):
        """Writes a single token's K and V vectors to the specified block and offset."""
        self.k_cache[physical_block_idx, token_offset] = k_tensor
        self.v_cache[physical_block_idx, token_offset] = v_tensor

# Initialize our allocator
kv_allocator = KVCacheAllocator(NUM_BLOCKS, BLOCK_SIZE, HEAD_DIM)
print(f"Initial Free Blocks: {len(kv_allocator.free_blocks)}")

Initial Free Blocks: 16


## 2. The Block Table
Each request (sequence) needs a way to map its continuous logical sequence of tokens to the scattered physical blocks. The `BlockTable` handles this mapping.

In [3]:
class SequenceState:
    def __init__(self, seq_id, allocator):
        self.seq_id = seq_id
        self.allocator = allocator
        
        # Block Table: maps logical block index to physical block index
        self.block_table = []
        
        # Track sequence length
        self.seq_len = 0
        
    def append_token(self, k_tensor, v_tensor):
        # Determine logical block and offset based on current sequence length
        logical_block_idx = self.seq_len // self.allocator.block_size
        token_offset = self.seq_len % self.allocator.block_size
        
        # If we need a new block, allocate it and update the block table
        if logical_block_idx >= len(self.block_table):
            new_physical_block = self.allocator.allocate_block()
            self.block_table.append(new_physical_block)
            print(f"[Seq {self.seq_id}] Allocated new Physical Block {new_physical_block} for Logical Block {logical_block_idx}")
            
        physical_block_idx = self.block_table[logical_block_idx]
        
        # Write to physical memory
        self.allocator.write_token(physical_block_idx, token_offset, k_tensor, v_tensor)
        self.seq_len += 1
        
        return physical_block_idx, token_offset

# Let's simulate a sequence generating 6 tokens (which requires 2 blocks)
seq = SequenceState(seq_id=1, allocator=kv_allocator)

# Dummy K, V vectors (shape: [HEAD_DIM])
for i in range(6):
    k = torch.ones(HEAD_DIM) * (i + 1)
    v = torch.ones(HEAD_DIM) * (i + 1)
    p_block, offset = seq.append_token(k, v)
    print(f"  Token {i} written to Physical Block {p_block}, Offset {offset}")

print(f"\nSequence Block Table: {seq.block_table}")
print(f"Remaining Free Blocks: {len(kv_allocator.free_blocks)}")

[Seq 1] Allocated new Physical Block 0 for Logical Block 0
  Token 0 written to Physical Block 0, Offset 0
  Token 1 written to Physical Block 0, Offset 1
  Token 2 written to Physical Block 0, Offset 2
  Token 3 written to Physical Block 0, Offset 3
[Seq 1] Allocated new Physical Block 1 for Logical Block 1
  Token 4 written to Physical Block 1, Offset 0
  Token 5 written to Physical Block 1, Offset 1

Sequence Block Table: [0, 1]
Remaining Free Blocks: 14


## 3. The PagedAttention Operation
In standard attention, we do `softmax(Q @ K.T) @ V`. 
Because K and V are scattered across physical blocks, PagedAttention fetches the blocks using the Block Table, reconstructing the contiguous K and V tensors (or doing the math block-by-block) before applying attention.

*Note: In production vLLM, this is a highly optimized custom CUDA kernel that computes attention directly on the scattered blocks without ever reconstructing the full contiguous tensor in memory. For this educational Python implementation, we will reconstruct it to show the logic.*

In [4]:
def paged_attention(query_tensor, seq_state, kv_allocator):
    """
    query_tensor: Shape [1, HEAD_DIM] - the query vector for the current token being generated
    seq_state: Contains the block_table and seq_len
    """
    # 1. Reconstruct the logical K and V tensors from the scattered physical blocks
    k_tensors = []
    v_tensors = []
    
    tokens_remaining = seq_state.seq_len
    
    for logical_idx, physical_idx in enumerate(seq_state.block_table):
        # How many tokens to read from this block?
        tokens_in_this_block = min(tokens_remaining, kv_allocator.block_size)
        
        # Fetch the K and V block slices
        k_block = kv_allocator.k_cache[physical_idx, :tokens_in_this_block, :]
        v_block = kv_allocator.v_cache[physical_idx, :tokens_in_this_block, :]
        
        k_tensors.append(k_block)
        v_tensors.append(v_block)
        
        tokens_remaining -= tokens_in_this_block
        
    # Concatenate blocks into a contiguous tensor (What the CUDA kernel avoids doing!)
    # Shape: [seq_len, HEAD_DIM]
    K = torch.cat(k_tensors, dim=0)
    V = torch.cat(v_tensors, dim=0)
    
    # 2. Standard Attention Computation
    # Scaled Dot-Product Attention: Softmax(Q * K^T / sqrt(d)) * V
    scores = torch.matmul(query_tensor, K.T) / (kv_allocator.head_dim ** 0.5)
    attention_weights = F.softmax(scores, dim=-1)
    
    # Output Shape: [1, HEAD_DIM]
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Let's test it! Generating the 7th token.
query = torch.randn(1, HEAD_DIM)

output, weights = paged_attention(query, seq, kv_allocator)

print(f"Query Shape: {query.shape}")
print(f"Attention Weights Shape: {weights.shape}")
print(f"Output Vector Shape: {output.shape}")
print(f"Attention Weights:\n{weights.detach().numpy()}")

Query Shape: torch.Size([1, 8])
Attention Weights Shape: torch.Size([1, 6])
Output Vector Shape: torch.Size([1, 8])
Attention Weights:
[[0.50981736 0.2538661  0.12641391 0.06294844 0.03134548 0.01560865]]


## Summary
This code demonstrates the exact mapping logic that happens during generation:
1. Tokens are grouped into `BLOCK_SIZE` chunks.
2. When a chunk fills up, a new physical block is pulled from the `KVCacheAllocator`.
3. The `BlockTable` tracks which physical block holds which logical chunk.
4. During attention computation, the `BlockTable` is used to fetch exactly the right physical blocks from GPU memory to compute the attention scores.

In a real deployment like vLLM, the `paged_attention` python function above is replaced by a custom CUDA/Triton kernel that performs the dot-product reductions in parallel across blocks without ever calling `torch.cat`.